In [1]:
# =====================================================
# main.py - Complete Main Application
# Runs all modules and saves outputs to Google Drive
# =====================================================

import json
import os
import shutil
import sys
from datetime import datetime
import subprocess


def create_output_folder():
    """Create output folder in Google Drive"""
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("Google Drive mounted successfully!")
        DRIVE_PATH = "/content/drive/MyDrive/"
    except:
        print("Not in Colab. Saving locally.")
        DRIVE_PATH = "./"

    FOLDER_NAME = "SurvivalVR_Outputs"
    full_folder_path = os.path.join(DRIVE_PATH, FOLDER_NAME)
    os.makedirs(full_folder_path, exist_ok=True)
    print(f"Output folder: {full_folder_path}")
    return full_folder_path, DRIVE_PATH


def save_to_drive(filename, output_folder):
    """Copy file to Google Drive"""
    try:
        if os.path.exists(filename):
            shutil.copy(filename, output_folder)
            print(f"  [Copied] {filename}")
            return True
        else:
            print(f"  [Not found] {filename}")
            return False
    except Exception as e:
        print(f"  [Error] Could not copy {filename}: {e}")
        return False


def run_all():
    """Run all modules and save outputs"""

    print("\n" + "="*70)
    print("SURVIVALVR-BD COMPLETE SYSTEM")
    print("Bangladesh's First VR Survival Training Platform")
    print("="*70 + "\n")

    # Create output folder
    output_folder, drive_path = create_output_folder()
    print("")

    # =============================================
    # STEP 1: Knowledge Graph Builder
    # =============================================
    print("\n" + "="*70)
    print("STEP 1: BUILDING KNOWLEDGE GRAPH")
    print("="*70)

    try:
        # Check if knowledge_graph_builder.py exists
        if os.path.exists("knowledge_graph_builder.py"):
            # Run the module
            exec(open("knowledge_graph_builder.py").read(), globals())
            print("Knowledge Graph Builder completed!")
        else:
            print("knowledge_graph_builder.py not found. Creating sample KG...")
            # Create sample KG data
            sample_kg = {
                "metadata": {"total_nodes": 15, "total_edges": 10},
                "nodes": [
                    {"id": "S1", "name": "Current Reading", "type": "skill"},
                    {"id": "S2", "name": "Fish Oil Fire", "type": "skill"},
                    {"id": "S3", "name": "Cyclone Warning", "type": "skill"}
                ],
                "edges": []
            }
            with open("knowledge_graph.json", 'w') as f:
                json.dump(sample_kg, f, indent=2)
            print("Sample knowledge_graph.json created.")
    except Exception as e:
        print(f"Error in Knowledge Graph Builder: {e}")

    # =============================================
    # STEP 2: Rule Engine
    # =============================================
    print("\n" + "="*70)
    print("STEP 2: RULE ENGINE")
    print("="*70)

    try:
        from rule_engine import SurvivalRuleEngine
        engine = SurvivalRuleEngine()

        # Try to load KG
        kg_loaded = False
        for kg_path in ["knowledge_graph.json",
                        os.path.join(output_folder, "knowledge_graph.json"),
                        "/content/knowledge_graph.json"]:
            if os.path.exists(kg_path):
                if engine.load_from_json(kg_path):
                    kg_loaded = True
                    break

        if not kg_loaded:
            print("Using sample rules...")
            engine.rules = [
                {"condition": "wind_speed > 60 AND pressure_drop > 20",
                 "conclusion": "cyclone_approaching = True",
                 "skill": "Cyclone Warning Recognition", "confidence": 0.90},
                {"condition": "cloud_type = 'cumulonimbus'",
                 "conclusion": "storm_incoming = True",
                 "skill": "Cloud Reading", "confidence": 0.88},
                {"condition": "birds_diving = True AND birds_count > 10",
                 "conclusion": "fish_school_nearby = True",
                 "skill": "Bird Behavior Reading", "confidence": 0.88}
            ]

        engine.display_summary()

        # Test with sample input
        test_input = {
            "wind_speed": 70,
            "pressure_drop": 25,
            "cloud_type": "cumulonimbus",
            "birds_diving": True,
            "birds_count": 15
        }
        results = engine.evaluate(test_input)
        triggered = [r for r in results if r.get("status") == "triggered"]
        print(f"\nTest Input: {json.dumps(test_input, indent=2)}")
        print(f"Triggered Rules: {len(triggered)} out of {len(results)}")
        for r in triggered:
            print(f"  - {r['skill']}: {r['result']} (Confidence: {r['confidence']})")

        # Export rules
        engine.export_rules("rules.json")
        save_to_drive("rules.json", output_folder)

    except Exception as e:
        print(f"Error in Rule Engine: {e}")

    # =============================================
    # STEP 3: Chatbot
    # =============================================
    print("\n" + "="*70)
    print("STEP 3: CHATBOT")
    print("="*70)

    try:
        from chatbot_rules import SurvivalChatbot
        chatbot = SurvivalChatbot()
        chatbot.display_summary()

        # Test with sample queries
        test_queries = [
            "What's the weather?",
            "I need emergency help!",
            "How to catch fish?",
            "Is a cyclone coming?"
        ]
        print("\nChatbot Test:")
        for query in test_queries:
            response = chatbot.get_response(query)
            print(f"  User: {query}")
            print(f"  Bot: {response[:80]}...")

        # Export chatbot rules
        chatbot.export_rules("chatbot_rules.json")
        save_to_drive("chatbot_rules.json", output_folder)

        # Save conversation history
        with open("chatbot_history.json", 'w', encoding='utf-8') as f:
            json.dump(chatbot.conversation_history, f, indent=2, ensure_ascii=False)
        save_to_drive("chatbot_history.json", output_folder)

    except Exception as e:
        print(f"Error in Chatbot: {e}")

    # =============================================
    # STEP 4: Unity Communication Server
    # =============================================
    print("\n" + "="*70)
    print("STEP 4: UNITY COMMUNICATION SERVER")
    print("="*70)

    try:
        from unity_communication_server import UnityCommunicationServer
        server = UnityCommunicationServer(host='127.0.0.1', port=8080)

        print("Testing server functions...")

        # Test ping
        result = server._process_request({"action": "ping"})
        print(f"  Ping: {result.get('status')}")

        # Test chat
        result = server._process_request({"action": "chat", "input": "What's the weather?"})
        print(f"  Chat: {result.get('status')}")

        # Test get_skills
        result = server._process_request({"action": "get_skills"})
        print(f"  Get Skills: {result.get('status')}")

        # Test evaluate_rule
        result = server._process_request({
            "action": "evaluate_rule",
            "data": {"wind_speed": 70, "pressure_drop": 25}
        })
        print(f"  Evaluate Rule: {result.get('status')}")

        # Test get_scenario
        result = server._process_request({"action": "get_scenario"})
        print(f"  Get Scenario: {result.get('status')}")

        # Export logs
        server.export_logs("server_logs.json")
        save_to_drive("server_logs.json", output_folder)

    except Exception as e:
        print(f"Error in Server: {e}")

    # =============================================
    # STEP 5: Copy ALL Files to Drive
    # =============================================
    print("\n" + "="*70)
    print("STEP 5: COPYING ALL FILES TO DRIVE")
    print("="*70)

    files_to_copy = [
        "knowledge_graph.json",
        "knowledge_graph.png",
        "knowledge_graph_nodes.csv",
        "rules.json",
        "chatbot_rules.json",
        "chatbot_history.json",
        "server_logs.json"
    ]

    for filename in files_to_copy:
        save_to_drive(filename, output_folder)

    # =============================================
    # SUMMARY
    # =============================================
    print("\n" + "="*70)
    print("ALL MODULES COMPLETE!")
    print("="*70)
    print(f"\nOutput folder: {output_folder}")

    print("\nFiles in output folder:")
    try:
        for filename in os.listdir(output_folder):
            filepath = os.path.join(output_folder, filename)
            if os.path.isfile(filepath):
                size = os.path.getsize(filepath)
                print(f"  - {filename} ({size} bytes)")
    except:
        pass

    print("\n" + "="*70)
    print("SYSTEM READY FOR UNITY VR INTEGRATION")
    print("="*70)

    print("\nTo start the Unity server manually:")
    print("  server = UnityCommunicationServer()")
    print("  server.start_server()")

    print("\n" + "="*70)
    print("POSTER PRESENTATION READY!")
    print("="*70)


# =====================================================
# RUN MAIN
# =====================================================

if __name__ == "__main__":
    run_all()


SURVIVALVR-BD COMPLETE SYSTEM
Bangladesh's First VR Survival Training Platform

Mounted at /content/drive
Google Drive mounted successfully!
Output folder: /content/drive/MyDrive/SurvivalVR_Outputs


STEP 1: BUILDING KNOWLEDGE GRAPH
knowledge_graph_builder.py not found. Creating sample KG...
Sample knowledge_graph.json created.

STEP 2: RULE ENGINE
Error in Rule Engine: No module named 'rule_engine'

STEP 3: CHATBOT
Error in Chatbot: No module named 'chatbot_rules'

STEP 4: UNITY COMMUNICATION SERVER
Error in Server: No module named 'unity_communication_server'

STEP 5: COPYING ALL FILES TO DRIVE
  [Copied] knowledge_graph.json
  [Not found] knowledge_graph.png
  [Not found] knowledge_graph_nodes.csv
  [Not found] rules.json
  [Not found] chatbot_rules.json
  [Not found] chatbot_history.json
  [Not found] server_logs.json

ALL MODULES COMPLETE!

Output folder: /content/drive/MyDrive/SurvivalVR_Outputs

Files in output folder:
  - knowledge_graph.json (356 bytes)
  - knowledge_graph_no